In [ ]:
import numpy as np

import os
from dotenv import load_dotenv
import sys

sys.path.append(os.path.abspath(os.path.join('..')))
import src.subvolume_utils as su
import src.correlation_tools as ct
import src.plotting as pg

load_dotenv()
data_dir = os.getenv('data_dir')
g_base = os.getenv('galaxy_base')
h_base = os.getenv('halo_base')
c_base = os.getenv('cluster_base')

In [ ]:
# calculates subvolume wp values for galaxy and halo data
def wp_subvolumes(galaxy, halo, h_var = '', h_min = 0, sub_ = 4, pimax = 100, nbins = 20,):
    hx_arr, hy_arr, hz_arr, h_sections, h_bin_id = su.subvolume_calc(halo, sub_ = sub_, var = h_var, min_ = h_min)
    gx_arr, gy_arr, gz_arr, g_sections, g_bin_id = su.subvolume_calc(galaxy, sub_ = sub_)
    all_wp = []
    all_rpavg = []

    rmin = 0.1
    rmax = 20.0
    rbins = np.logspace(np.log10(rmin), np.log10(rmax), nbins + 1)

    for i in range(0, (sub_**3)):
        sub_vol_gx = gx_arr[g_bin_id[g_sections[i - 1]: g_sections[i]]]
        sub_vol_gy = gy_arr[g_bin_id[g_sections[i - 1]: g_sections[i]]]
        sub_vol_gz = gz_arr[g_bin_id[g_sections[i - 1]: g_sections[i]]]
        sub_vol_hx = hx_arr[h_bin_id[h_sections[i - 1]: h_sections[i]]]
        sub_vol_hy = hy_arr[h_bin_id[h_sections[i - 1]: h_sections[i]]]
        sub_vol_hz = hz_arr[h_bin_id[h_sections[i - 1]: h_sections[i]]]
        
        boxsize = max(np.max(sub_vol_gx), np.max(sub_vol_gx)) - min(np.min(sub_vol_gx), np.min(sub_vol_gx))

        wp_, rpavg_ = ct.wp_pairs_cross(sub_vol_gx, sub_vol_gy, sub_vol_gz, sub_vol_hx, sub_vol_hy, sub_vol_hz, pimax = pimax, bins = rbins, boxsize=boxsize)
        all_wp.append(wp_)
        all_rpavg.append(rpavg_)
    
    return all_wp, all_rpavg


In [ ]:
galaxies = f"{g_base}base_c000_ph000/z0p300/model_hod000000/gals.fit"
halos = f"{h_base}base_c000_ph000/z0p300/halos_3e+12.fit"
wp_calc, rp_calc = wp_subvolumes(galaxies, halos, h_var = 'mass', h_min = 10**14)
for i in range (1, 25):
    galaxies = f"{g_base}base_c000_ph{i:03d}/z0p300/model_hod000000/gals.fit"
    halos = f"{h_base}base_c000_ph{i:03d}/z0p300/halos_3e+12.fit"
    wp_sim, rp_sim = wp_subvolumes(galaxies, halos, h_var = 'mass', h_min = 10**14)
    wp_calc = np.concatenate((wp_calc, wp_sim))
    rp_calc = np.concatenate((rp_calc, rp_sim))
#np.savez("wp_subvol_no_cross.npz", rp_all = rp_, wp_all = wp_)

In [ ]:
print(np.min(wp_calc))

In [ ]:
data = np.load(f'{data_dir}wp_subvol_no_cross.npz')
rp_ = data['rp_all']
wp_ = data['wp_all']
data.files

In [ ]:
pg.plot_results(rp_, wp_, y_max = 5000, y_min = 6.5)

In [ ]:
rpavg_mean, wp_mean, stdev = ct.error_data(rp_, wp_)
pg.wp_vs_rpavg(rpavg_mean, wp_mean, stdev, y_max = 5000, y_min = 6.5)   

In [ ]:
frac_err = pg.get_frac_err(stdev, wp_mean)
pg.plot_frac_err([rpavg_mean], [frac_err],labels = ['mean'])